### Additive Metadata Embeddings
Use item-user interaction with Review signals, category, and price

In [ ]:
import pandas as pd

# Load the final filtered dataset
review_data = pd.read_json('../data/processed/review_data.jsonl', lines=True)
metadata = pd.read_json('../data/processed/metadata.jsonl', lines=True)

print(f"Loaded {len(review_data)} reviews and {len(metadata)} metadata records")
print("Review data columns:", review_data.columns.tolist())
print("Metadata columns:", metadata.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


# Assume df has ['reviewerID', 'asin', 'overall', 'reviewTime']
ratings_df = review_data[['user_id', 'parent_asin', 'rating', 'reviewTime']].copy()
metadata_df = metadata[['parent_asin', 'main_category', 'average_rating', 'rating_number', 'price']].copy()

ratings_df.head()

# Step 1 - Data Preparation

import torch

# Map users/items to integer IDs.
user2idx = {u: i for i, u in enumerate(ratings_df['user_id'].unique())}
item2idx = {i: j for j, i in enumerate(ratings_df['parent_asin'].unique())}

# Add the encoded columns
ratings_df['user_idx'] = ratings_df['user_id'].map(user2idx)
ratings_df['item_idx'] = ratings_df['parent_asin'].map(item2idx)

users = torch.tensor(ratings_df['user_idx'].values)
items = torch.tensor(ratings_df['item_idx'].values)
ratings = torch.tensor(ratings_df['rating'].values, dtype=torch.float32)

## Metadata Preparation
metadata_df['price'] = pd.to_numeric(metadata_df['price'], errors='coerce')
metadata_df['average_rating'] = pd.to_numeric(metadata_df['average_rating'], errors='coerce').fillna(0)
metadata_df['rating_number'] = pd.to_numeric(metadata_df['rating_number'], errors='coerce').fillna(0).astype(int)

# Transform numeric fields (log + scale where it makes sense)
metadata_df['price_log'] = np.log1p(metadata_df['price'])
metadata_df['rating_number_log'] = np.log1p(metadata_df['rating_number'])


scalers = {}
for col in ['price_log', 'average_rating', 'rating_number_log']:
    scaler = StandardScaler()
    metadata_df[col + '_scaled'] = scaler.fit_transform(metadata_df[[col]])
    scalers[col] = scaler

# Map main_category to integer IDs
main_categories = metadata_df['main_category'].fillna('Unknown').astype(str)
cat2idx = {cat: idx+1 for idx, cat in enumerate(main_categories.unique())}
cat2idx['<unk>'] = 0
metadata_df['main_cat_idx'] = main_categories.map(lambda c: cat2idx.get(c, 0))


# Merge on parent_asin
merged_df = ratings_df.merge(
    metadata_df[['parent_asin', 'main_cat_idx', 
                 'price_log_scaled', 'average_rating_scaled', 'rating_number_log_scaled']],
    on='parent_asin',
    how='left'
)



In [ ]:
print(merged_df.head())
print("Users:", merged_df['user_id'].nunique())
print("Items:", merged_df['parent_asin'].nunique())
print("Categories:", len(cat2idx))

In [ ]:
# Train test split
# For each user, keep their last review as test, and earlier ones as train.
#sort by time
merged_df = merged_df.sort_values(by=['user_id', 'reviewTime'])

#Split
test_df = merged_df.groupby('user_id').tail(1)
train_df = merged_df.drop(test_df.index)

train_users = torch.tensor(train_df['user_idx'].values)
train_items = torch.tensor(train_df['item_idx'].values)
train_ratings = torch.tensor(train_df['rating'].values, dtype=torch.float32)
train_main_cat = torch.tensor(train_df['main_cat_idx'].values)
train_prices = torch.tensor(train_df['price_log_scaled'].values, dtype=torch.float32)
train_avg_ratings = torch.tensor(train_df['average_rating_scaled'].values, dtype=torch.float32)
train_number_ratings = torch.tensor(train_df['rating_number_log_scaled'].values, dtype=torch.float32)

test_users = torch.tensor(test_df['user_idx'].values)
test_items = torch.tensor(test_df['item_idx'].values)
test_ratings = torch.tensor(test_df['rating'].values, dtype=torch.float32)
test_main_cat = torch.tensor(test_df['main_cat_idx'].values)
test_prices = torch.tensor(test_df['price_log_scaled'].values, dtype=torch.float32)
test_avg_ratings = torch.tensor(test_df['average_rating_scaled'].values, dtype=torch.float32)
test_number_ratings = torch.tensor(test_df['rating_number_log_scaled'].values, dtype=torch.float32)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AdditiveHybridMF(nn.Module):


    def __init__(self,
                 n_users,
                 n_items,
                 n_main_cats,
                 emb_dim=64,
                 use_avg_rating=True,
                 use_rating_count=True,
                 use_price=True,
                 dropout=0.0):
        super().__init__()
        self.emb_dim = emb_dim

        # main embeddings
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.cat_emb  = nn.Embedding(n_main_cats, emb_dim)   # 0 reserved for <unk>

        # numeric feature projections (scalar -> embedding)
        self.use_price = use_price
        self.use_avg_rating = use_avg_rating
        self.use_rating_count = use_rating_count

        if self.use_price:
            self.price_proj = nn.Linear(1, emb_dim, bias=True)
        else:
            self.price_proj = None

        if self.use_avg_rating:
            self.avg_proj = nn.Linear(1, emb_dim, bias=True)
        else:
            self.avg_proj = None

        if self.use_rating_count:
            self.count_proj = nn.Linear(1, emb_dim, bias=True)
        else:
            self.count_proj = None

        # biases
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)

        # optional dropout on item vector
        self.dropout = nn.Dropout(dropout) if dropout > 0 else None

        # initialization (small normal)
        self._init_weights()

    def _init_weights(self):
        std = 0.01
        for emb in (self.user_emb, self.item_emb, self.cat_emb):
            nn.init.normal_(emb.weight, mean=0.0, std=std)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)
        # Linear proj init
        if self.price_proj is not None:
            nn.init.xavier_uniform_(self.price_proj.weight)
            nn.init.zeros_(self.price_proj.bias)
        if self.avg_proj is not None:
            nn.init.xavier_uniform_(self.avg_proj.weight)
            nn.init.zeros_(self.avg_proj.bias)
        if self.count_proj is not None:
            nn.init.xavier_uniform_(self.count_proj.weight)
            nn.init.zeros_(self.count_proj.bias)

    def forward_item_vector(self, item_idx, main_cat_idx, price_val=None,
                            avg_rating_val=None, rating_count_val=None):

        v_item = self.item_emb(item_idx)           # (B, D)
        v_cat  = self.cat_emb(main_cat_idx)        # (B, D)
        parts = [v_item, v_cat]

        if self.price_proj is not None and price_val is not None:
            # ensure shape (B,1)
            pv = price_val.view(-1, 1).float()
            v_price = self.price_proj(pv)         # (B, D)
            parts.append(v_price)

        if self.avg_proj is not None and avg_rating_val is not None:
            av = avg_rating_val.view(-1, 1).float()
            v_avg = self.avg_proj(av)
            parts.append(v_avg)

        if self.count_proj is not None and rating_count_val is not None:
            cv = rating_count_val.view(-1, 1).float()
            v_count = self.count_proj(cv)
            parts.append(v_count)

        item_vec = sum(parts)  # additive combination

        if self.dropout is not None:
            item_vec = self.dropout(item_vec)

        return item_vec

    def score(self, user_idx, item_idx, main_cat_idx, price_val=None,
              avg_rating_val=None, rating_count_val=None):

        u = self.user_emb(user_idx)                 # (B, D)
        item_vec = self.forward_item_vector(item_idx, main_cat_idx,
                                            price_val, avg_rating_val, rating_count_val)  # (B, D)
        dot = (u * item_vec).sum(dim=-1)            # (B,)
        b_u = self.user_bias(user_idx).squeeze(-1)  # (B,)
        b_i = self.item_bias(item_idx).squeeze(-1)  # (B,)
        return dot + b_u + b_i

    def forward(self, user_idx, item_idx, main_cat_idx, price_val=None,
                avg_rating_val=None, rating_count_val=None):

        return self.score(user_idx, item_idx, main_cat_idx, price_val, avg_rating_val, rating_count_val)



In [ ]:
import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
import numpy as np
from time import time

# --- Embedding size bases (use max index + 1 to cover full index space incl 0) --- #
n_users = int(train_df['user_idx'].max()) + 1
n_items = int(train_df['item_idx'].max()) + 1
n_main_cats = int(train_df['main_cat_idx'].max()) + 1  # includes 0 = <unk>

# Also check if test set has indices beyond training range
test_max_users = int(test_df['user_idx'].max()) + 1
test_max_items = int(test_df['item_idx'].max()) + 1
test_max_cats = int(test_df['main_cat_idx'].max()) + 1

if test_max_users > n_users or test_max_items > n_items or test_max_cats > n_main_cats:
    print("WARNING: Test set contains indices beyond training range!")
    print(f"Train ranges: users={n_users}, items={n_items}, cats={n_main_cats}")
    print(f"Test ranges: users={test_max_users}, items={test_max_items}, cats={test_max_cats}")
    # Use max from both train and test to be safe
    n_users = max(n_users, test_max_users)
    n_items = max(n_items, test_max_items)
    n_main_cats = max(n_main_cats, test_max_cats)

print(f"Final embedding sizes: n_users={n_users} n_items={n_items} n_main_cats={n_main_cats}")

model = AdditiveHybridMF(
    n_users=n_users,
    n_items=n_items,
    n_main_cats=n_main_cats,
    emb_dim=64,
    use_avg_rating=True,
    use_rating_count=True,
    use_price=True,
    dropout=0.1
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print("Device:", device)

# --- Build item-level metadata arrays (use ONLY train data to avoid leakage) --- #
item_cat = np.zeros(n_items, dtype=np.int64)
item_price = np.zeros(n_items, dtype=np.float32)
item_avg = np.zeros(n_items, dtype=np.float32)
item_count = np.zeros(n_items, dtype=np.float32)

meta_source = (
    train_df[['item_idx','main_cat_idx','price_log_scaled','average_rating_scaled','rating_number_log_scaled']]
    .drop_duplicates('item_idx')
)
for _, row in meta_source.iterrows():
    i = int(row.item_idx)
    if i < n_items:  # safety
        item_cat[i] = int(row.main_cat_idx) if pd.notna(row.main_cat_idx) else 0
        item_price[i] = float(row.price_log_scaled) if pd.notna(row.price_log_scaled) else 0.0
        item_avg[i] = float(row.average_rating_scaled) if pd.notna(row.average_rating_scaled) else 0.0
        item_count[i] = float(row.rating_number_log_scaled) if pd.notna(row.rating_number_log_scaled) else 0.0

# Check for any remaining NaN values and fill with zeros
item_price = np.nan_to_num(item_price, nan=0.0)
item_avg = np.nan_to_num(item_avg, nan=0.0) 
item_count = np.nan_to_num(item_count, nan=0.0)

# Convert to tensors & move to device once
item_cat_t = torch.from_numpy(item_cat).to(device)
item_price_t = torch.from_numpy(item_price).to(device)
item_avg_t = torch.from_numpy(item_avg).to(device)
item_count_t = torch.from_numpy(item_count).to(device)

# Training interaction tensors (kept on CPU until indexing then moved in batch)
u_arr = torch.from_numpy(train_df['user_idx'].values.astype(np.int64))
pos_item_arr = torch.from_numpy(train_df['item_idx'].values.astype(np.int64))
pos_cat_arr = torch.from_numpy(train_df['main_cat_idx'].values.astype(np.int64))
pos_price_arr = torch.from_numpy(train_df['price_log_scaled'].values.astype(np.float32))
pos_avg_arr   = torch.from_numpy(train_df['average_rating_scaled'].values.astype(np.float32))
pos_count_arr = torch.from_numpy(train_df['rating_number_log_scaled'].values.astype(np.float32))

n_train = u_arr.shape[0]
print("Training triples:", n_train)

# Check for any missing values in training data and handle them
print("NaN check in training arrays:")
print(f"pos_price_arr has NaN: {torch.isnan(pos_price_arr).any().item()}")
print(f"pos_avg_arr has NaN: {torch.isnan(pos_avg_arr).any().item()}")
print(f"pos_count_arr has NaN: {torch.isnan(pos_count_arr).any().item()}")

# Replace any NaN values with 0
pos_price_arr = torch.nan_to_num(pos_price_arr, nan=0.0)
pos_avg_arr = torch.nan_to_num(pos_avg_arr, nan=0.0)
pos_count_arr = torch.nan_to_num(pos_count_arr, nan=0.0)

# Build user->set(positive items) for better negative sampling (train only)
user_pos = {}
for u,i in zip(u_arr.numpy(), pos_item_arr.numpy()):
    user_pos.setdefault(int(u), set()).add(int(i))

# Optimizer (Adam)
decay, no_decay = [], []
for name, p in model.named_parameters():
    if not p.requires_grad: 
        continue
    if p.ndim == 1 or name.endswith(".bias"):
        no_decay.append(p)
    else:
        decay.append(p)
optimizer = optim.AdamW([{'params': decay, 'weight_decay': 1e-5},
                         {'params': no_decay, 'weight_decay': 0.0}], lr=5e-4)

# BPR loss
softplus = nn.Softplus()

def bpr_loss(pos_scores, neg_scores):
    return softplus(neg_scores - pos_scores).mean()

def sample_neg(user_batch, n_items, user_pos_sets, device):
    """Sample one negative per user avoiding known positives (simple rejection)."""
    B = user_batch.size(0)
    neg = torch.randint(0, n_items, (B,), device=device)
    for idx, u in enumerate(user_batch.tolist()):
        tries = 0
        while neg[idx].item() in user_pos_sets.get(u, ()) and tries < 10:
            neg[idx] = torch.randint(0, n_items, (1,), device=device)
            tries += 1
    return neg

def train_bpr(model, epochs=5, batch_size=2048, l2_lambda=0.0):
    n = n_train
    indices = np.arange(n)
    model.train()
    for epoch in range(1, epochs+1):
        
        np.random.shuffle(indices)
        epoch_loss = 0.0
        seen = 0
        t0 = time()
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start+batch_size]
            if len(batch_idx) == 0:
                continue
            user = u_arr[batch_idx].to(device)
            pos_item = pos_item_arr[batch_idx].to(device)
            pos_cat  = pos_cat_arr[batch_idx].to(device)
            pos_price = pos_price_arr[batch_idx].to(device)
            pos_avg   = pos_avg_arr[batch_idx].to(device)
            pos_count = pos_count_arr[batch_idx].to(device)

            neg_item = sample_neg(user, n_items, user_pos, device)
            neg_cat = item_cat_t[neg_item]
            neg_price = item_price_t[neg_item]
            neg_avg = item_avg_t[neg_item]
            neg_count = item_count_t[neg_item]

            optimizer.zero_grad()
            pos_scores = model.score(user, pos_item, pos_cat, pos_price, pos_avg, pos_count)
            neg_scores = model.score(user, neg_item, neg_cat, neg_price, neg_avg, neg_count)
            loss = bpr_loss(pos_scores, neg_scores)
            if l2_lambda > 0:
                # light embedding norm penalty
                l2 = (model.user_emb(user).pow(2).mean() +
                      model.item_emb(pos_item).pow(2).mean() +
                      model.item_emb(neg_item).pow(2).mean())
                loss = loss + l2_lambda * l2
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch_idx)
            seen += len(batch_idx)
        avg = epoch_loss / max(1, seen)
        print(f"Epoch {epoch}/{epochs}  BPR_Loss={avg:.5f}  time={time()-t0:.1f}s")
    print("Training complete.")

# Run training
train_bpr(model=model, epochs=5, batch_size=2048, l2_lambda=1e-6)


In [ ]:
# This cell is now redundant - metadata arrays are already built in the training cell above
# Keeping this as a placeholder but the actual work is done in the training cell

print("Metadata arrays already built in previous cell:")
print(f"item_cat_t shape: {item_cat_t.shape}")
print(f"item_price_t shape: {item_price_t.shape}")
print(f"item_avg_t shape: {item_avg_t.shape}")
print(f"item_count_t shape: {item_count_t.shape}")
print(f"All arrays are on device: {item_cat_t.device}")

In [ ]:
import math
import numpy as np

# Filter out cold start users from evaluation
train_users_set = set(train_df['user_idx'].unique())
test_df_filtered = test_df[test_df['user_idx'].isin(train_users_set)]

# Rebuild metadata arrays cleanly
item_cat = np.zeros(n_items, dtype=np.int64)
item_price = np.zeros(n_items, dtype=np.float32)
item_avg = np.zeros(n_items, dtype=np.float32)
item_count = np.zeros(n_items, dtype=np.float32)

meta_source = train_df[['item_idx','main_cat_idx','price_log_scaled','average_rating_scaled','rating_number_log_scaled']].drop_duplicates('item_idx')

for _, row in meta_source.iterrows():
    i = int(row.item_idx)
    if i < n_items:
        item_cat[i] = int(row.main_cat_idx) if pd.notna(row.main_cat_idx) else 0
        item_price[i] = float(row.price_log_scaled) if pd.notna(row.price_log_scaled) else 0.0
        item_avg[i] = float(row.average_rating_scaled) if pd.notna(row.average_rating_scaled) else 0.0
        item_count[i] = float(row.rating_number_log_scaled) if pd.notna(row.rating_number_log_scaled) else 0.0

item_price = np.nan_to_num(item_price, nan=0.0)
item_avg = np.nan_to_num(item_avg, nan=0.0) 
item_count = np.nan_to_num(item_count, nan=0.0)

item_cat_t = torch.from_numpy(item_cat).to(device)
item_price_t = torch.from_numpy(item_price).to(device)
item_avg_t = torch.from_numpy(item_avg).to(device)
item_count_t = torch.from_numpy(item_count).to(device)

# Ranking metrics
def precision_at_k(ranked_indices, positive_index, k):
    if k == 0: return 0.0
    topk = ranked_indices[:k]
    hit = 1.0 if positive_index in topk else 0.0
    return hit / k

def hit_at_k(ranked_indices, positive_index, k):
    return 1.0 if positive_index in ranked_indices[:k] else 0.0

def ndcg_at_k(ranked_indices, positive_index, k):
    topk = ranked_indices[:k]
    if positive_index in topk:
        rank = topk.index(positive_index) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0

def reciprocal_rank(ranked_indices, positive_index):
    for rank, idx in enumerate(ranked_indices, start=1):
        if idx == positive_index:
            return 1.0 / rank
    return 0.0

# Build user training items mapping
user_train_items = {}
for _, row in train_df.iterrows():
    u = int(row.user_idx)
    item = int(row.item_idx)
    if u not in user_train_items:
        user_train_items[u] = set()
    user_train_items[u].add(item)

# Evaluation
model.eval()
all_items_tensor = torch.arange(n_items, device=device)

Ks = (5, 10, 20)
metrics_acc = { 'precision': {k: [] for k in Ks},
                'hit': {k: [] for k in Ks},
                'ndcg': {k: [] for k in Ks},
                'rr': [] }

with torch.no_grad():
    for _, row in test_df_filtered.iterrows():
        u = int(row.user_idx)
        true_item = int(row.item_idx)
        
        if u >= n_users or true_item >= n_items:
            continue
        
        # Score all items and exclude training items
        user_tensor = torch.full((n_items,), u, device=device, dtype=torch.long)
        scores = model.score(user_tensor, all_items_tensor, item_cat_t, item_price_t, item_avg_t, item_count_t)
        
        # Mask training items by setting their scores to -inf
        train_items = user_train_items.get(u, set())
        for train_item in train_items:
            scores[train_item] = float('-inf')
        
        # Get top-k using torch.topk
        _, top_indices = torch.topk(scores, k=min(max(Ks), n_items), largest=True)
        ranked_eval_items = top_indices.cpu().numpy().tolist()

        # Calculate metrics
        for k in Ks:
            metrics_acc['precision'][k].append(precision_at_k(ranked_eval_items, true_item, k))
            metrics_acc['hit'][k].append(hit_at_k(ranked_eval_items, true_item, k))
            metrics_acc['ndcg'][k].append(ndcg_at_k(ranked_eval_items, true_item, k))
        metrics_acc['rr'].append(reciprocal_rank(ranked_eval_items, true_item))

results = {}
for k in Ks:
    results[f'Precision@{k}'] = float(np.mean(metrics_acc['precision'][k]))
    results[f'HitRate@{k}'] = float(np.mean(metrics_acc['hit'][k]))
    results[f'NDCG@{k}'] = float(np.mean(metrics_acc['ndcg'][k]))
results['MRR'] = float(np.mean(metrics_acc['rr']))

# Results table
print("Ranking Evaluation Results:")
print("="*50)
header = "K".ljust(8) + "Precision".ljust(12) + "HitRate".ljust(12) + "NDCG".ljust(12)
print(header)
print("-" * 44)
for k in Ks:
    print(str(k).ljust(8) +
          f"{results[f'Precision@{k}']:.4f}".ljust(12) +
          f"{results[f'HitRate@{k}']:.4f}".ljust(12) +
          f"{results[f'NDCG@{k}']:.4f}".ljust(12))
print(f"\nMRR: {results['MRR']:.4f}")